# Career Mentor — FT-III Evaluation Notebook

**Purpose:** evaluate the prototype's deterministic recommendation logic using synthetic student profiles in Google Colab.

**Important:** These tests measure software behavior on generated cases. They do not establish job-placement effectiveness, production-scale performance, or the quality of Gemini responses. Run all cells in order; use the generated metrics in your report only after you have run the notebook.


## 1. Environment and reproducibility

The notebook uses only Python's standard library and synthetic profiles. It does not require an API key or call an external AI service. This makes the evaluation reproducible and avoids sharing personal data.


In [ ]:
import random
import time
import statistics
from collections import Counter

SEED = 42
random.seed(SEED)
print("Python evaluation ready. Random seed:", SEED)


## 2. Role-skill map and recommendation rules

This is a compact evaluation copy of the same role-gap principle used in the Streamlit app. If you modify `ROLE_SKILLS` or the app recommendation engine, update this cell too or import the latest project code from GitHub.


In [ ]:
ROLE_SKILLS = {
    "Software Engineer": ["Python", "Problem Solving", "Data Structures", "SQL", "Git", "APIs", "Testing", "Cloud Basics"],
    "Data Analyst": ["SQL", "Python", "Statistics", "Data Visualization", "Excel", "Communication", "Power BI"],
    "AI/ML Engineer": ["Python", "Statistics", "Machine Learning", "Data Handling", "Model Evaluation", "Git", "APIs", "Cloud Basics"],
    "Cloud Engineer": ["Linux", "Networking", "Cloud Basics", "Git", "Security Basics", "Scripting", "Containers"],
    "Frontend Developer": ["HTML/CSS", "JavaScript", "Accessibility", "APIs", "Git", "Testing", "UI Design"],
}
ACTIONABLE_SKILLS = set(skill for skills in ROLE_SKILLS.values() for skill in skills)

def recommend(profile, limit=5):
    required = ROLE_SKILLS[profile["target_role"]]
    current = set(profile["skills"])
    gaps = [skill for skill in required if skill not in current]
    # Transparent deterministic policy: prioritize missing required skills in role-map order.
    return gaps[:limit]

def coverage(profile):
    required = ROLE_SKILLS[profile["target_role"]]
    current = set(profile["skills"])
    return sum(skill in current for skill in required) / len(required) if required else 0.0


## 3. Generate synthetic test profiles

The generated profiles contain only skill labels and target roles. No real student records are used.


In [ ]:
profiles = []
for i in range(200):
    role = random.choice(list(ROLE_SKILLS))
    required = ROLE_SKILLS[role]
    skills = [s for s in required if random.random() < 0.45]
    profiles.append({"case_id": i, "target_role": role, "skills": skills})

print("Synthetic profiles:", len(profiles))
print("Role distribution:", dict(Counter(p["target_role"] for p in profiles)))


## 4. Functional evaluation metrics

- **Recommendation validity:** fraction of recommendations that belong to the selected target role's required skills and are missing from the student's selected skills.
- **Gap coverage@5:** fraction of missing required skills surfaced in the first five recommendations, averaged over profiles with at least one gap.
- **Mean recommendation runtime:** average local Python time to calculate recommendations for one profile. This is a microbenchmark on the current notebook runtime, not a web request latency measurement.
- **Role readiness coverage:** mean fraction of required skills currently selected. This is a profile coverage indicator, not hiring probability.


In [ ]:
valid_total = 0
rec_total = 0
gap_coverages = []
readiness_values = []
times_ms = []

for profile in profiles:
    required = set(ROLE_SKILLS[profile["target_role"]])
    current = set(profile["skills"])
    gaps = required - current
    start = time.perf_counter()
    recs = recommend(profile, limit=5)
    elapsed_ms = (time.perf_counter() - start) * 1000
    times_ms.append(elapsed_ms)

    valid_total += sum((r in required and r not in current) for r in recs)
    rec_total += len(recs)
    if gaps:
        gap_coverages.append(len(set(recs) & gaps) / len(gaps))
    readiness_values.append(coverage(profile))

recommendation_validity = valid_total / rec_total if rec_total else 1.0
mean_gap_coverage_at_5 = statistics.mean(gap_coverages) if gap_coverages else 1.0
mean_readiness = statistics.mean(readiness_values) if readiness_values else 0.0
mean_runtime_ms = statistics.mean(times_ms) if times_ms else 0.0
p95_runtime_ms = sorted(times_ms)[min(len(times_ms)-1, int(0.95 * len(times_ms)))] if times_ms else 0.0

print(f"Recommendation validity: {recommendation_validity:.1%}")
print(f"Mean gap coverage@5: {mean_gap_coverage_at_5:.1%}")
print(f"Mean role-skill coverage: {mean_readiness:.1%}")
print(f"Mean local recommendation runtime: {mean_runtime_ms:.6f} ms/profile")
print(f"Approximate p95 local runtime: {p95_runtime_ms:.6f} ms/profile")


## 5. Edge cases and assertions

These checks verify empty skill lists, fully covered roles, and recommendation membership.


In [ ]:
# Empty profile skills should produce valid, role-relevant recommendations.
empty_profile = {"target_role": "Software Engineer", "skills": []}
assert recommend(empty_profile, 5)
assert all(skill in ROLE_SKILLS[empty_profile["target_role"]] for skill in recommend(empty_profile, 5))

# If all role skills are present, there are no skill gaps.
complete_role = {"target_role": "Data Analyst", "skills": ROLE_SKILLS["Data Analyst"].copy()}
assert coverage(complete_role) == 1.0
assert recommend(complete_role) == []

# Every recommendation must be a missing skill for the selected role.
for profile in profiles:
    required = set(ROLE_SKILLS[profile["target_role"]])
    current = set(profile["skills"])
    assert all(item in required and item not in current for item in recommend(profile, 5))

print("All edge-case assertions passed.")


## 6. Results table and interpretation

Use the actual values printed above. Do not report this local runtime as Streamlit cloud response time. For a stronger evaluation, add manual browser tests, deploy the app, record cold-start and normal page-load timings, and test persistence after an app restart using a managed database.


In [ ]:
results = {
    "Synthetic profiles": len(profiles),
    "Recommendation validity (%)": round(recommendation_validity * 100, 2),
    "Mean gap coverage@5 (%)": round(mean_gap_coverage_at_5 * 100, 2),
    "Mean role-skill coverage (%)": round(mean_readiness * 100, 2),
    "Mean local runtime (ms/profile)": round(mean_runtime_ms, 6),
    "Approx. p95 local runtime (ms/profile)": round(p95_runtime_ms, 6),
    "Edge-case assertions": "PASS",
}
for metric, value in results.items():
    print(f"{metric}: {value}")


## 7. FT-III technical evaluation notes

**Strengths:** deterministic and explainable recommendations; no API key needed for baseline testing; tests use generated profiles; evaluation can be reproduced.

**Limitations:** synthetic profiles do not represent real student outcomes; recommendations are skill-gap heuristics rather than a trained predictive model; runtime is measured in the notebook and is not a deployment benchmark; Gemini output quality is not evaluated by this notebook; SQLite on ephemeral hosting may not provide durable storage.

**Next evaluation step:** after Streamlit deployment, manually test each of the four sections, capture screenshots, and record the exact deployed URL and date. Do not claim a deployment is complete until the URL opens and the app has been tested.
